# 3. SupportSense — RAG Assistant & Answer Grounding
## Objective
Run questions through the retrieval-augmented answer pipeline and inspect answer text, citations, and abstention behaviour.

The default mode is a deterministic extractive fallback. Optional OpenAI generation is supported by the project, but requires explicit configuration and an API key. Do not commit credentials.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from rag_core import SupportKnowledgeBase, baseline_answer, rag_answer, evaluate_answer

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
docs = json.loads((ROOT / "data" / "company_docs.json").read_text(encoding="utf-8"))
questions = json.loads((ROOT / "data" / "evaluation_questions.json").read_text(encoding="utf-8"))
kb = SupportKnowledgeBase(docs)

In [ ]:
# Inspect a single question end to end.
q = questions[0]
retrieved = kb.retrieve(q["question"], k=3)
result = rag_answer(q, retrieved)
print("Question:", q["question"])
print("Answer:", result["answer"])
print("Citations:", result["citations"])
print("Abstained:", result["abstained"])
display(pd.DataFrame([{"rank": i+1, "doc_id": d["doc_id"], "score": round(d["score"], 3), "title": d["title"]} for i,d in enumerate(retrieved)]))

## Batch review: citations and answerability
The checks below use labelled expected documents and expected terms from the synthetic benchmark. These are useful proxies for reviewing examples; they are not a substitute for blinded human evaluation.

In [ ]:
review_rows = []
for q in questions:
    hits = kb.retrieve(q["question"], k=3)
    result = rag_answer(q, hits)
    review_rows.append({
        "question_id": q["question_id"],
        "question": q["question"],
        "answerable": q.get("answerable", True),
        "answer": result["answer"],
        "citations": ", ".join(result["citations"]),
        "abstained": result["abstained"],
        "expected_doc": q.get("expected_doc"),
        "expected_doc_cited": bool(q.get("expected_doc") and q.get("expected_doc") in result["citations"])
    })
review_df = pd.DataFrame(review_rows)
display(review_df)

## Safety and grounding checklist
- The answer should be supported by approved passages, not model memory.
- Citations should point to documents that actually support the answer.
- Questions outside the knowledge base should trigger an abstention or escalation path.
- Never expose secrets, one-time codes, or private customer information.
- For a production system, add human review, prompt-injection tests, access control, logging redaction, and source freshness checks.